# Code the explanations with two LLM judges

The free-form explanations of one model (`outputs/<MODEL>/explanations.csv`, written by `run_model.ipynb`) are coded by Claude Haiku 5.5 and GPT-6 Luna (OpenRouter) with the five yes/no codes and the prompt in `configs/judges.toml`, the same for every model; the judges do not see the condition. Results, in the same folder: `judge_labels.jsonl`, one row per (explanation, judge), plus the agreement between the judges and the codes by condition; finished rows are skipped. The OpenRouter key is read from `.env`.

In [ ]:
MODEL = "gemma-4-12b"  # a table in configs/models.toml
N_QUESTIONS = None  # None = all 598

In [ ]:
import hashlib, json, os, threading, tomllib
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm

ROOT = Path.cwd().parent
CFG = tomllib.loads((ROOT / "configs/judges.toml").read_text())
SOURCE = tomllib.loads((ROOT / "configs/full_run.toml").read_text())
CODES = list(CFG["codes"])
JUDGES = CFG["judge_models"]
EXP_DIR = ROOT / "outputs" / MODEL
OUT = EXP_DIR / "judge_labels.jsonl"

expl = pd.read_csv(EXP_DIR / "explanations.csv")
options = pd.read_parquet(ROOT / "data-private/processed/emotion_questions.parquet").set_index("question_id")
expl["key"] = expl.question_id + "|" + expl.condition
expl = expl[expl.order.isin(sorted(expl.order.unique())[:N_QUESTIONS])]


def options_text(row):
    texts = {L: options.loc[row.question_id, f"option_{L}"].strip() for L in "ABCD"} | {"E": SOURCE["option_e"]}
    return "\n".join(f"{shown}. {texts[original]}" for shown, original in zip("ABCDE", row.option_order))


expl["options_shown"] = [options_text(r) for r in expl.itertuples()]
expl["answer_text"] = [r.options_shown.split("\n")["ABCDE".index(r.answer_shown)] for r in expl.itertuples()]
print(f"{expl.question_id.nunique()} questions, {len(expl)} explanations; judges: {JUDGES} -> {OUT.relative_to(ROOT)}")

## Judges

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(ROOT / ".env")
client = OpenAI(base_url=CFG["api_base_url"], api_key=os.environ["OPENROUTER_API_KEY"])
SYSTEM = CFG["prompts"]["system"].format(codes="\n".join(f"- {c}: {d}" for c, d in CFG["codes"].items()))
PROMPT_SHA = hashlib.sha256(SYSTEM.encode()).hexdigest()[:12]
SCHEMA = {"type": "object",
          "properties": {"rationale": {"type": "string"}, **{c: {"type": "boolean"} for c in CODES}},
          "required": ["rationale"] + CODES, "additionalProperties": False}


def item_text(row):
    return CFG["prompts"]["item"].format(question=row.question.strip(), options=row.options_shown,
                                         answer=row.answer_text, explanation=row.explanation)


def judge(row, model):
    extra = {"seed": CFG["seed"]} if model in CFG["seed_models"] else {}
    response = client.chat.completions.create(
        model=model, max_tokens=CFG["max_tokens"],
        messages=[{"role": "system", "content": SYSTEM}, {"role": "user", "content": item_text(row)}],
        response_format={"type": "json_schema", "json_schema": {"name": "codes", "strict": True, "schema": SCHEMA}},
        extra_body={"reasoning": {"effort": CFG["reasoning_effort"]}}, **extra,
    )
    choice = response.choices[0]
    labels = json.loads(choice.message.content)
    assert set(labels) == {"rationale"} | set(CODES), labels
    details = getattr(response.usage, "completion_tokens_details", None)
    return {"key": row.key, "model": model, **labels, "finish_reason": choice.finish_reason,
            "response_model": response.model, "prompt_tokens": response.usage.prompt_tokens,
            "completion_tokens": response.usage.completion_tokens,
            "reasoning_tokens": getattr(details, "reasoning_tokens", None), "prompt_sha": PROMPT_SHA}


def done():
    return {(r["key"], r["model"]) for r in map(json.loads, OUT.open())} if OUT.exists() else set()


print(SYSTEM)
print("\n===== example item")
print(item_text(expl.iloc[0]))

## Run

In [ ]:
lock, errors = threading.Lock(), []

def work(task):
    row, model = task
    try:
        result = judge(row, model)
    except Exception as e:
        errors.append((row.key, model, repr(e)[:200]))
        return
    with lock, OUT.open("a") as f:
        f.write(json.dumps(result) + "\n")

finished = done()
tasks = [(row, model) for model in JUDGES for row in expl.itertuples() if (row.key, model) not in finished]
with ThreadPoolExecutor(CFG["n_workers"]) as pool:
    list(tqdm(pool.map(work, tasks), total=len(tasks), unit="label"))
print(f"done {len(tasks) - len(errors)}, failed {len(errors)}")
for e in errors[:10]:
    print(" ", *e)

## Check

In [ ]:
labels = pd.read_json(OUT, lines=True)
labels = labels[labels.key.isin(expl.key)]
display(labels.groupby("model")[["finish_reason", "reasoning_tokens"]].agg(lambda s: s.value_counts(dropna=False).to_dict()))
labels["codes"] = labels[CODES].apply(lambda r: ", ".join(c for c in CODES if r[c]) or "-", axis=1)
side = labels.pivot(index="key", columns="model", values="codes")
side.columns = [m.split("/")[1] for m in side.columns]
pd.set_option("display.max_colwidth", None)
display(expl.head(30).set_index("key")[["explanation"]].join(side))

## Status

In [ ]:
labels = pd.read_json(OUT, lines=True)
status = labels.groupby("model").agg(rows=("key", "nunique"), prompt_tokens=("prompt_tokens", "sum"),
                                      completion_tokens=("completion_tokens", "sum"))
status["cost_usd"] = (status.prompt_tokens * 0.10 + status.completion_tokens * 0.50) / 1e6
print(f"{len(expl)} explanations; prompt versions: {labels.prompt_sha.value_counts().to_dict()}")
display(status)

## Agreement between the judges

In [ ]:
from sklearn.metrics import cohen_kappa_score

labels = pd.read_json(OUT, lines=True, dtype={"prompt_sha": str})
labels = labels[labels.key.isin(expl.key)]
wide = {m: labels[labels.model == m].set_index("key")[CODES].astype(bool).reindex(expl.key) for m in JUDGES}
a, b = wide[JUDGES[0]], wide[JUDGES[1]]
agreement = pd.DataFrame({"agree": (a == b).mean(), "kappa": [cohen_kappa_score(a[c], b[c]) for c in CODES],
                          **{f"yes_{m.split('/')[1]}": wide[m].sum() for m in JUDGES}})
agreement.to_csv(EXP_DIR / "agreement.csv")
print("explanations with any disagreement:", (a != b).any(axis=1).sum(), "of", len(expl))
agreement.round(3)

## Codes by condition

% of explanations where both judges give the code.

In [ ]:
conditions = [c for c in SOURCE["conditions"] if c in set(expl.condition)]
condition = expl.condition.values
by_condition = (a & b).groupby(condition).mean().reindex(conditions) * 100
by_condition["judges_differ"] = (a != b).any(axis=1).groupby(condition).mean().reindex(conditions) * 100
by_condition.to_csv(EXP_DIR / "codes_by_condition.csv")
audio = expl.explanation.str.contains(r"\baudio", case=False)
print('explanations mentioning "audio":', audio.groupby(condition).sum().reindex(conditions).to_dict())
by_condition.round(1)